# Notebook 03 — Classificação de Temas em DOIS EIXOS

**Sprint 2 — Lei e Política** · *redesign da clusterização*

## Por que reescrever
A versão anterior aplicava **TF-IDF + K-Means (K=10) sobre TODA a base** e tinha dois
defeitos estruturais comprovados (ver `auditoria_clusters.py`):

1. **Cluster catch-all** — 62,1% das 22.106 proposições caíam num único balde
   ("Políticas Públicas e Programas Sociais"). O K-Means colapsava no maior balde
   tudo sem vocabulário distintivo.
2. **Tensão de taxonomia** — os 10 rótulos misturavam dois critérios incompatíveis:
   *forma* (licença, requerimento, radiodifusão, crédito) e *assunto* (saúde, educação,
   penal). Um "requerimento de informação ao Ministro da Saúde" podia cair na forma OU
   no assunto: ambiguidade que está no **rótulo**, não no dado.

## A correção: dois eixos independentes
- **Eixo A — Natureza (regras, sem ML).** Classifica o *instrumento* por regex
  (`src/classificacao/natureza.py`). ~36% da base é "ruído de forma" que nunca deveria
  ter ido ao K-Means temático. Não competem com o assunto.
- **Eixo B — Tema (ML, só sobre as substantivas A3).** TF-IDF + K-Means **apenas** nas
  proposições que efetivamente alteram o ordenamento. Aqui o K-Means trabalha sobre
  conteúdo real.

`tema_cidadao` final = rótulo fixo da forma (A0/A1/A2) **ou** tema descoberto (A3).

In [1]:
import sys
sys.path.insert(0, '..')

import re
import unicodedata
import logging

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from src.db import buscar_todos, upsert_proposicoes
from src.classificacao.natureza import classificar_natureza, ROTULO_NATUREZA

logging.basicConfig(level=logging.WARNING, format='%(asctime)s [%(levelname)s] %(message)s')
print('Módulos carregados.')

Módulos carregados.


## 1. Carregar o corpus de ementas

In [2]:
props = buscar_todos('proposicoes', 'id,id_externo,casa,ementa')
df = pd.DataFrame(props)
print(f'Proposições no banco: {len(df)}')

df = df[df['ementa'].notna()].copy()
df['ementa'] = df['ementa'].astype(str)
print(f'Com ementa: {len(df)}')
print(df['casa'].value_counts())

Proposições no banco: 22106
Com ementa: 22106
casa
camara    11762
senado    10344
Name: count, dtype: int64


## 2. EIXO A — Natureza (regras determinísticas)

A natureza vem do módulo reutilizável `src/classificacao/natureza.py` (testado em
`tests/test_natureza.py`). Só as **A3 substantivas** seguem para o Eixo B.

In [3]:
nat = df['ementa'].map(classificar_natureza)
df['natureza_codigo'] = nat.map(lambda x: x[0])
df['eixo'] = np.where(df['natureza_codigo'] == 'A3_Substantiva', 'tema', 'forma')

dist = df['natureza_codigo'].value_counts()
print('=== Distribuição por natureza (Eixo A) ===')
for k, n in dist.items():
    print(f'{n:6d}  {100*n/len(df):5.1f}%  {k}')

n_a3 = int((df['natureza_codigo'] == 'A3_Substantiva').sum())
print(f'\nSubstantivas (A3) → Eixo B: {n_a3} ({100*n_a3/len(df):.1f}%)')
print(f'Ruído de forma (A0+A1+A2): {100*(len(df)-n_a3)/len(df):.1f}% — fora do K-Means')

=== Distribuição por natureza (Eixo A) ===
 14148   64.0%  A3_Substantiva
  6107   27.6%  A0_Administrativa
  1705    7.7%  A2_Radiodifusao
   146    0.7%  A1_Orcamentaria

Substantivas (A3) → Eixo B: 14148 (64.0%)
Ruído de forma (A0+A1+A2): 36.0% — fora do K-Means


## 3. Limpeza do texto (só A3)

Minúsculas → sem acentos → só letras → descarte de tokens curtos e *stopwords*.
Além das stopwords PT, estendemos com **jargão jurídico** (`altera`, `lei`, `art`,
`dispõe`, `dá outras providências`…) e **nomes de meses** — estes últimos
dominavam o TF-IDF por causa das datas em toda citação de lei
("Lei nº X, de DD de *dezembro* de AAAA") e achatavam os clusters.

In [4]:
def remover_acentos(t):
    nfkd = unicodedata.normalize('NFKD', t)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

STOPWORDS_PT = set('''a o as os um uma uns umas de do da dos das em no na nos nas por pelo
pela pelos pelas com sem sob sobre para pra ate entre contra desde e ou mas que se como
quando porque pois ja nao sim ao aos este esta estes estas esse essa esses essas isto isso
aquele aquela aquilo seu sua seus suas dele dela deles delas meu minha nosso nossa ele ela
eles elas eu tu voce nos vos lhe lhes me te foi ser sao era sera tem ter havia mais menos
muito pouco todo toda todos todas outro outra outros outras mesmo mesma qual quais onde seja
sejam tambem apenas cada ainda assim entao'''.split())

# Jargão jurídico onipresente — não distingue temas
STOPWORDS_JURIDICAS = set('''lei leis art arts artigo artigos paragrafo inciso incisos alinea
dispoe dispor dispondo dispoem altera alteracao alterar alterada institui instituir estabelece
estabelecer providencias outras revoga revogacao vigencia dar acrescenta acrescentar inclui
inclusao incluir modifica modificar denomina denominacao denominada autoriza autorizacao cria
criacao criar federal nacional numero decreto medida provisoria projeto proposta emenda
constituicao codigo normas norma regula regulamenta regulamentacao define fixa fixar concede
referente relativo relativa seguinte seguintes redacao termos quanto vista efeito efeitos data
partir vigor publicacao consolidacao consolidadas'''.split())

MESES = 'janeiro fevereiro marco abril maio junho julho agosto setembro outubro novembro dezembro'.split()

STOPWORDS = {remover_acentos(w) for w in STOPWORDS_PT | STOPWORDS_JURIDICAS | set(MESES)}

def limpar(texto):
    texto = remover_acentos(texto.lower())
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    return ' '.join(t for t in texto.split() if len(t) >= 3 and t not in STOPWORDS)

a3 = df[df['natureza_codigo'] == 'A3_Substantiva'].copy()
a3['texto_limpo'] = a3['ementa'].map(limpar)
a3 = a3[a3['texto_limpo'].str.len() > 0].copy()
print(f'Substantivas com texto após limpeza: {len(a3)}')
print('Exemplo:', a3['texto_limpo'].iloc[0][:140])

Substantivas com texto após limpeza: 14091
Exemplo: dispositivos prever responsabilidade penal pessoas juridicas casos infracao cometida decisao representante legal contratual orgao colegiado 


## 4. TF-IDF (só A3)

`sublinear_tf=True` aplica escala logarítmica à frequência, amortecendo termos
dominantes — essencial num corpus de boilerplate jurídico repetido.

In [5]:
vectorizer = TfidfVectorizer(
    sublinear_tf=True, max_df=0.30, min_df=8, ngram_range=(1, 2), max_features=6000,
)
X = vectorizer.fit_transform(a3['texto_limpo'])
print(f'Matriz TF-IDF (só A3): {X.shape[0]} documentos × {X.shape[1]} termos')

Matriz TF-IDF (só A3): 14091 documentos × 5509 termos


## 5. Escolha de K — cotovelo (inertia) + silhueta

Varremos `K = 4..20`. A **inertia** (soma das distâncias intra-cluster) cai de forma
suave — sem cotovelo nítido, sintoma honesto de estrutura global fraca em texto legal
curto. A **silhueta** é baixa em toda a faixa (~0,02) e cresce devagar (clusters pequenos
e densos puxam a média). Acompanhamos também o **% do maior cluster**: a métrica que de
fato importa aqui, porque o defeito a vencer é a concentração num catch-all.

In [6]:
sample = min(5000, X.shape[0])
res = []
for k in range(4, 21):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X)
    sil = silhouette_score(X, lab, sample_size=sample, random_state=42)
    maior = 100 * pd.Series(lab).value_counts().max() / len(lab)
    res.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil, 'maior_pct': maior})
curva = pd.DataFrame(res)

print('  K   inertia    silhueta   maior_cluster%')
imax = curva['inertia'].max()
for _, r in curva.iterrows():
    barra = '#' * int(round(34 * r['inertia'] / imax))
    print(f"  {int(r['k']):2d}  {r['inertia']:8.1f}  {r['silhouette']:.4f}   {r['maior_pct']:5.1f}  {barra}")

  K   inertia    silhueta   maior_cluster%
   4   13769.1  0.0051    92.3  ##################################
   5   13694.6  0.0086    84.6  ##################################
   6   13685.9  0.0091    80.5  ##################################
   7   13616.3  0.0092    76.5  ##################################
   8   13548.1  0.0128    66.5  #################################
   9   13496.5  0.0136    66.2  #################################
  10   13467.1  0.0143    56.7  #################################
  11   13412.8  0.0156    56.7  #################################
  12   13382.1  0.0155    60.3  #################################
  13   13345.9  0.0171    62.9  #################################
  14   13319.3  0.0178    52.0  #################################
  15   13288.0  0.0184    50.0  #################################
  16   13242.5  0.0190    54.0  #################################
  17   13234.7  0.0194    57.5  #################################
  18   13169.5  0.0209    53.

### Decisão de K — justificativa para a banca

- A inertia não tem cotovelo nítido (queda suave): o critério do cotovelo é **inconclusivo**
  aqui, o que é coerente com texto legal curto e boilerplate compartilhado.
- A silhueta é baixa (~0,02) em toda a faixa — estrutura de cluster global fraca. Subir K
  só eleva a silhueta marginalmente porque cria micro-clusters muito coesos.
- O **maior cluster** cai de ~85% (K=4) para ~45% (K=20). Como o objetivo do redesign é
  **quebrar o catch-all**, escolhemos **K=20**: minimiza o cluster dominante mantendo
  rótulos legíveis. Mesmo assim o maior cluster fica em ~45% (> 35% do critério de aceite):
  documentamos isso como **limitação inerente** — ementas curtas com vocabulário genérico
  ("dispõe sobre", "política", "programa") não têm sinal para separar mais sem explodir K.
  O ganho real do Eixo A já foi remover os ~36% de ruído de forma: o resíduo agora é
  **tematicamente homogêneo** (políticas públicas gerais), não mais uma mistura forma×tema.

In [7]:
K = 20
km = KMeans(n_clusters=K, random_state=42, n_init=10)
a3['tema_cluster'] = km.fit_predict(X)
maior = 100 * a3['tema_cluster'].value_counts().max() / len(a3)
print(f'K={K}  maior cluster = {maior:.1f}% das substantivas')

K=20  maior cluster = 46.6% das substantivas


## 6. Top-termos por cluster (base da nomeação cidadã)

In [8]:
termos = vectorizer.get_feature_names_out()
ordem = km.cluster_centers_.argsort()[:, ::-1]

def top_termos(c, n=10):
    return [termos[i] for i in ordem[c, :n]]

print(f'=== Top-10 termos por cluster (K={K}) ===\n')
for c in sorted(range(K), key=lambda c: -(a3['tema_cluster'] == c).sum()):
    n = int((a3['tema_cluster'] == c).sum())
    print(f"Cluster {c:2d} ({n:5d}, {100*n/len(a3):4.1f}%): {', '.join(top_termos(c))}")

=== Top-10 termos por cluster (K=20) ===

Cluster  4 ( 6573, 46.6%): programa, civil, protecao, pessoas, saude, obrigatoriedade, consumidor, uso, servicos, brasil
Cluster  8 (  951,  6.7%): publicos, territorio, publicas, instituicoes, ensino, obrigatoriedade, politicas, servicos, servicos publicos, politicas publicas
Cluster  5 (  907,  6.4%): penal, crime, crimes, pena, processo penal, processo, tipificar, penal tipificar, aumento pena, aumento
Cluster  1 (  574,  4.1%): politica, prevencao, protecao, incentivo, diretrizes, politica protecao, politica prevencao, politica incentivo, urbana, desenvolvimento
Cluster 12 (  538,  3.8%): susta, trabalho, portaria, susta portaria, trabalho clt, clt, resolucao, aprovada, clt aprovada, susta resolucao
Cluster 18 (  489,  3.5%): estado, municipio, reconhece, capital, manifestacao, cultura, manifestacao cultura, titulo, titulo capital, rio
Cluster 19 (  412,  2.9%): educacao, basica, educacao basica, bases educacao, diretrizes bases, bases, dir

## 7. Nomeação cidadã dos clusters

Mapa `cluster → nome` derivado dos top-termos acima (execução determinística,
`random_state=42`). O cluster dominante recebe o rótulo honesto **"Outras Políticas
Públicas"**. A célula seguinte **verifica** o alinhamento imprimindo nome ↔ top-termos:
se a base mudar, basta reconferir aqui.

In [9]:
NOMES_EIXO_B = {
    4:  'Outras Políticas Públicas',                  # resíduo (~47%)
    8:  'Administração e Serviços Públicos',
    5:  'Direito Penal e Crimes',
    1:  'Políticas de Prevenção e Incentivo',
    12: 'Sustação de Atos do Executivo',
    18: 'Homenagens e Patrimônio Cultural',
    19: 'Educação',
    10: 'Segurança Pública',
    15: 'Previdência e Assistência Social',
    11: 'Tributação e Reforma Tributária',
    3:  'Saúde e SUS',
    13: 'Pessoa com Deficiência e Idoso',
    9:  'Criança e Adolescente',
    17: 'Violência Doméstica e Direitos da Mulher',
    6:  'Datas Comemorativas',
    7:  'Operações de Crédito de Municípios',
    2:  'Proteção de Crianças (Digital e Sexual)',
    16: 'Transtorno do Espectro Autista (TEA)',
    14: 'Trânsito e Veículos',
    0:  'Indicações de Autoridades (Sabatinas)',
}
assert set(NOMES_EIXO_B) == set(range(K)), 'mapa de nomes não cobre todos os clusters'

print('=== Verificação nome ↔ top-termos ===\n')
for c in sorted(range(K), key=lambda c: -(a3['tema_cluster'] == c).sum()):
    print(f"[{NOMES_EIXO_B[c]:46s}] {', '.join(top_termos(c, 6))}")

a3['tema_cidadao'] = a3['tema_cluster'].map(NOMES_EIXO_B)

=== Verificação nome ↔ top-termos ===

[Outras Políticas Públicas                     ] programa, civil, protecao, pessoas, saude, obrigatoriedade
[Administração e Serviços Públicos             ] publicos, territorio, publicas, instituicoes, ensino, obrigatoriedade
[Direito Penal e Crimes                        ] penal, crime, crimes, pena, processo penal, processo
[Políticas de Prevenção e Incentivo            ] politica, prevencao, protecao, incentivo, diretrizes, politica protecao
[Sustação de Atos do Executivo                 ] susta, trabalho, portaria, susta portaria, trabalho clt, clt
[Homenagens e Patrimônio Cultural              ] estado, municipio, reconhece, capital, manifestacao, cultura
[Educação                                      ] educacao, basica, educacao basica, bases educacao, diretrizes bases, bases
[Segurança Pública                             ] publica, seguranca, seguranca publica, administracao publica, administracao, profissionais seguranca
[Previdência e As

## 8. Rótulo unificado e gravação

`tema_cidadao` final:
- **A0/A1/A2** → rótulo fixo da forma (de `ROTULO_NATUREZA`), `tema_cluster = NULL`, `eixo='forma'`.
- **A3** → tema do Eixo B, `tema_cluster` preenchido, `eixo='tema'`.

Gravamos via `upsert_proposicoes` (chave `id_externo, casa`), atualizando só as colunas
de classificação.

In [10]:
# Rótulo de forma para A0/A1/A2 (não passaram pelo Eixo B)
df['tema_cidadao'] = df['natureza_codigo'].map(ROTULO_NATUREZA)
df['tema_cluster'] = np.nan

# Sobrescreve as A3 com tema e cluster do Eixo B
a3_idx = a3.set_index(['id_externo', 'casa'])
df = df.set_index(['id_externo', 'casa'])
df.loc[a3_idx.index, 'tema_cidadao'] = a3_idx['tema_cidadao']
df.loc[a3_idx.index, 'tema_cluster'] = a3_idx['tema_cluster']
df = df.reset_index()

print('=== tema_cidadao final (todas as proposições) ===')
vc = df['tema_cidadao'].value_counts(dropna=False)
for nome, n in vc.items():
    print(f'{n:6d}  {100*n/len(df):5.1f}%  {nome}')

=== tema_cidadao final (todas as proposições) ===
  6573   29.7%  Outras Políticas Públicas
  6107   27.6%  Requerimentos e Atos Internos
  1705    7.7%  Outorgas de Radiodifusão
   951    4.3%  Administração e Serviços Públicos
   907    4.1%  Direito Penal e Crimes
   574    2.6%  Políticas de Prevenção e Incentivo
   538    2.4%  Sustação de Atos do Executivo
   489    2.2%  Homenagens e Patrimônio Cultural
   412    1.9%  Educação
   403    1.8%  Segurança Pública
   388    1.8%  Previdência e Assistência Social
   379    1.7%  Tributação e Reforma Tributária
   341    1.5%  Saúde e SUS
   339    1.5%  Pessoa com Deficiência e Idoso
   270    1.2%  Criança e Adolescente
   263    1.2%  Violência Doméstica e Direitos da Mulher
   244    1.1%  Operações de Crédito de Municípios
   244    1.1%  Datas Comemorativas
   242    1.1%  Proteção de Crianças (Digital e Sexual)
   214    1.0%  Transtorno do Espectro Autista (TEA)
   174    0.8%  Trânsito e Veículos
   146    0.7%  Indicações d

In [11]:
registros = [
    {
        'id_externo': int(r['id_externo']),
        'casa': r['casa'],
        'natureza_codigo': r['natureza_codigo'],
        'eixo': r['eixo'],
        'tema_cluster': (int(r['tema_cluster']) if pd.notna(r['tema_cluster']) else None),
        'tema_cidadao': r['tema_cidadao'],
    }
    for _, r in df.iterrows()
]
print(f'Registros a gravar: {len(registros)}')
total = upsert_proposicoes(registros)
print(f'Proposições atualizadas: {total}')

Registros a gravar: 22106


Proposições atualizadas: 22106


## 9. Sanidade (lê de volta do banco)

In [12]:
verif = pd.DataFrame(buscar_todos('proposicoes',
        'casa,natureza_codigo,eixo,tema_cluster,tema_cidadao'))

print('=== natureza_codigo ===')
print(verif['natureza_codigo'].value_counts(dropna=False))

print('\n=== Invariante: A0/A1/A2 nunca têm tema_cluster ===')
forma = verif[verif['eixo'] == 'forma']
print('formas com tema_cluster preenchido (deve ser 0):',
      int(forma['tema_cluster'].notna().sum()))

print('\n=== Invariante: toda A3 tem tema_cluster ===')
tema = verif[verif['eixo'] == 'tema']
print('A3 sem tema_cluster (deve ser 0):', int(tema['tema_cluster'].isna().sum()))

print('\n=== tema_cidadao por eixo ===')
print(verif.groupby('eixo')['tema_cidadao'].nunique().rename('n_temas'))

=== natureza_codigo ===
natureza_codigo
A3_Substantiva       14148
A0_Administrativa     6107
A2_Radiodifusao       1705
A1_Orcamentaria        146
Name: count, dtype: int64

=== Invariante: A0/A1/A2 nunca têm tema_cluster ===
formas com tema_cluster preenchido (deve ser 0): 0

=== Invariante: toda A3 tem tema_cluster ===
A3 sem tema_cluster (deve ser 0): 57

=== tema_cidadao por eixo ===
eixo
forma     3
tema     20
Name: n_temas, dtype: int64
